# Seção 2 — Prompt engineering e saídas controladas

In [1]:
from pathlib import Path
import os
import json
import re
from datetime import datetime

import pandas as pd
from gpt4all import GPT4All

## 1. Caminhos do projeto

In [2]:
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_RAW = ROOT_DIR / "data" / "raw"
OUTPUT_DIR = ROOT_DIR / "outputs" / "avaliacoes"
MODEL_DIR = ROOT_DIR / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Diretório raiz:", ROOT_DIR)
print("Corpus:", DATA_RAW)
print("Modelos:", MODEL_DIR)
print("Saídas:", OUTPUT_DIR)

Diretório raiz: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment
Corpus: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\data\raw
Modelos: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\models
Saídas: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes


## 2. Carregamento de trecho do corpus

In [3]:
def ler_arquivo(caminho: Path) -> str:
    return caminho.read_text(encoding="utf-8")

arquivos_corpus = sorted(DATA_RAW.glob("*.txt"))

print(f"Total de arquivos encontrados: {len(arquivos_corpus)}")
for arquivo in arquivos_corpus:
    print("-", arquivo.name)

arquivo_teste = DATA_RAW / "01_contexto_geral_1836_1936.txt"
texto_teste = ler_arquivo(arquivo_teste)

contexto = texto_teste[:2800]

print("\nArquivo usado como contexto:", arquivo_teste.name)
print("Tamanho do contexto:", len(contexto), "caracteres")
print("\nPrévia do contexto:\n")
print(contexto[:1000])

Total de arquivos encontrados: 12
- 01_contexto_geral_1836_1936.txt
- 02_reino_unido_primeira_industrializacao.txt
- 03_ferrovias_carvao_aco.txt
- 04_segunda_revolucao_industrial.txt
- 05_estados_unidos_capitalismo_industrial.txt
- 06_alemanha_industria_ciencia_estado.txt
- 07_japao_modernizacao_meiji.txt
- 08_imperialismo_materias_primas.txt
- 09_trabalho_urbano_movimentos_operarios.txt
- 10_primeira_guerra_economia_industrial.txt
- 11_crise_1929_capitalismo.txt
- 12_urss_planejamento_industrializacao.txt

Arquivo usado como contexto: 01_contexto_geral_1836_1936.txt
Tamanho do contexto: 2800 caracteres

Prévia do contexto:

Título: Contexto geral da industrialização e da formação da economia moderna, 1836-1936

Período principal: 1836-1936

Temas:
industrialização; economia moderna; tecnologia; capitalismo industrial; urbanização; imperialismo; guerras industriais; crise econômica; modelos de desenvolvimento

Países e atores:
Reino Unido; Estados Unidos; Alemanha; Japão; União Soviéti

## 3. Inicialização do GPT4All

In [4]:
MODEL_NAME = "Phi-3-mini-4k-instruct.Q4_0.gguf"

try:
    gpus_disponiveis = GPT4All.list_gpus()
except Exception as erro:
    gpus_disponiveis = []
    print("Não foi possível listar GPUs:", erro)

print("GPUs disponíveis:", gpus_disponiveis)

device_escolhido = "gpu" if len(gpus_disponiveis) > 0 else "cpu"

print("Dispositivo escolhido:", device_escolhido)

try:
    model = GPT4All(
        model_name=MODEL_NAME,
        model_path=str(MODEL_DIR),
        allow_download=True,
        n_threads=os.cpu_count(),
        n_ctx=2048,
        device=device_escolhido
    )

except Exception as erro:
    print("Falha ao carregar em GPU. Tentando carregar em CPU.")
    print("Erro:", erro)

    device_escolhido = "cpu"

    model = GPT4All(
        model_name=MODEL_NAME,
        model_path=str(MODEL_DIR),
        allow_download=True,
        n_threads=os.cpu_count(),
        n_ctx=2048,
        device=device_escolhido
    )

print("Modelo carregado:", MODEL_NAME)
print("Executando em:", device_escolhido)

GPUs disponíveis: ['kompute:NVIDIA GeForce RTX 4070']
Dispositivo escolhido: gpu
Modelo carregado: Phi-3-mini-4k-instruct.Q4_0.gguf
Executando em: gpu


## 4. Função auxiliar de geração

In [5]:
SYSTEM_PROMPT = """
Você é um assistente histórico especializado em industrialização e formação da economia moderna.
Responda em português brasileiro.
Siga exatamente o formato solicitado pelo usuário.
Quando for solicitado JSON, responda apenas com JSON válido, sem markdown e sem explicações externas.
Não invente fontes ou fatos que não estejam no contexto fornecido.
"""

def gerar_resposta(prompt: str, max_tokens: int = 500, temp: float = 0.1) -> str:
    """Gera uma resposta usando GPT4All."""
    prompt_completo = f"""
{SYSTEM_PROMPT}

{prompt}

Resposta:
"""

    with model.chat_session():
        resposta = model.generate(
            prompt_completo,
            max_tokens=max_tokens,
            temp=temp,
            top_k=40,
            top_p=0.9,
            repeat_penalty=1.15
        )

    return resposta.strip()


## 5. Esquema da saída controlada

In [6]:
CAMPOS_OBRIGATORIOS = [
    "pergunta",
    "resposta",
    "atores_historicos",
    "fontes_recuperadas",
    "nivel_confianca",
    "limitacoes"
]

pergunta_teste = "Quais processos históricos tornam o período 1836-1936 relevante para a economia moderna?"

print("Pergunta de teste:")
print(pergunta_teste)
print("\nCampos obrigatórios esperados:")
print(CAMPOS_OBRIGATORIOS)

Pergunta de teste:
Quais processos históricos tornam o período 1836-1936 relevante para a economia moderna?

Campos obrigatórios esperados:
['pergunta', 'resposta', 'atores_historicos', 'fontes_recuperadas', 'nivel_confianca', 'limitacoes']


## 6. Funções de parsing, validação e tratamento de erro

In [7]:
def extrair_json(texto: str) -> dict:
    """Extrai o primeiro objeto JSON encontrado em uma resposta textual."""
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if not match:
        raise ValueError("Nenhum objeto JSON foi encontrado na resposta.")

    trecho_json = match.group(0)
    return json.loads(trecho_json)


def validar_json_resposta(dados: dict) -> tuple[bool, list[str]]:
    """Valida campos obrigatórios, tipos e valores esperados."""
    erros = []

    for campo in CAMPOS_OBRIGATORIOS:
        if campo not in dados:
            erros.append(f"Campo ausente: {campo}")

    if "pergunta" in dados and str(dados["pergunta"]).strip() != pergunta_teste:
        erros.append("O campo pergunta deve reproduzir exatamente a pergunta de teste.")

    if "resposta" in dados and not str(dados["resposta"]).strip():
        erros.append("O campo resposta não pode estar vazio.")

    if "limitacoes" in dados and not str(dados["limitacoes"]).strip():
        erros.append("O campo limitacoes não pode estar vazio.")

    if "atores_historicos" in dados and not isinstance(dados["atores_historicos"], list):
        erros.append("O campo atores_historicos deve ser uma lista.")

    if "fontes_recuperadas" in dados:
        if not isinstance(dados["fontes_recuperadas"], list):
            erros.append("O campo fontes_recuperadas deve ser uma lista.")
        elif arquivo_teste.name not in dados["fontes_recuperadas"]:
            erros.append(f"O campo fontes_recuperadas deve conter {arquivo_teste.name}.")

    if "nivel_confianca" in dados:
        niveis_validos = ["baixo", "médio", "medio", "alto"]
        if str(dados["nivel_confianca"]).lower() not in niveis_validos:
            erros.append("O campo nivel_confianca deve ser baixo, médio ou alto.")

    return len(erros) == 0, erros


def processar_saida(nome_tecnica: str, resposta_bruta: str) -> dict:
    """Aplica parsing e validação, retornando um registro de avaliação."""
    try:
        dados = extrair_json(resposta_bruta)
        valido, erros = validar_json_resposta(dados)
        return {
            "tecnica": nome_tecnica,
            "json_valido": valido,
            "erros": erros,
            "dados": dados,
            "resposta_bruta": resposta_bruta
        }
    except Exception as e:
        return {
            "tecnica": nome_tecnica,
            "json_valido": False,
            "erros": [str(e)],
            "dados": None,
            "resposta_bruta": resposta_bruta
        }


## 7. Técnica 1 — Zero-shot prompting

Nesta técnica, o modelo recebe apenas a tarefa e o formato esperado, sem exemplos.

In [8]:
prompt_zero_shot = f"""
Responda à pergunta com base no contexto fornecido.

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta_teste}
</pergunta>

Responda somente com JSON válido.
Não use markdown.
Não use bloco ```json.
Não escreva nenhuma explicação fora do JSON.
A resposta deve começar com {{ e terminar com }}.

O JSON deve conter exatamente estes campos:
{{
  "pergunta": "{pergunta_teste}",
  "resposta": "texto da resposta",
  "atores_historicos": ["lista de atores históricos citados"],
  "fontes_recuperadas": ["{arquivo_teste.name}"],
  "nivel_confianca": "alto",
  "limitacoes": "limitações da resposta"
}}
"""

resposta_zero_shot = gerar_resposta(prompt_zero_shot, max_tokens=500, temp=0.1)

print(resposta_zero_shot)


{
  "pergunta": "Quais processos históricos tornam o período 1836-1936 relevante para a economia moderna?",
  "resposta": "O período de 1836-1936 é marcado pela industrialização, expansão do capitalismo e transformação da organização econômica e social. A consolidação das fábricas, o uso crescente de energia fóssil, a expansão ferroviária e a formação de mercados nacionais integrados foram processos cruciais que influenciaram profundamente as bases da economia moderna.",
  "atores_historicos": ["Reino Unido", "Estados Unidos", "Alemanha", "Japão"],
  "fontes_recuperadas": ["01_contexto_geral_1836_1936.txt"],
  "nivel_confianca": "alto",
  "limitacoes": "Não inclui todos os processos históricos relevantes, mas aborda alguns dos mais significativos."
}


In [9]:
avaliacao_zero_shot = processar_saida("zero-shot", resposta_zero_shot)

print("JSON válido:", avaliacao_zero_shot["json_valido"])
print("Erros:", avaliacao_zero_shot["erros"])

JSON válido: True
Erros: []


## 8. Técnica 2 — Prompt estruturado com papel, contexto, tarefa e formato

Nesta técnica, o prompt separa explicitamente papel do modelo, contexto, tarefa e formato esperado.

In [10]:
prompt_estruturado = f"""
PAPEL:
Você é um assistente histórico especializado em industrialização e formação da economia moderna.

CONTEXTO:
<contexto>
{contexto}
</contexto>

TAREFA:
Responda à pergunta usando somente o contexto fornecido.

PERGUNTA:
{pergunta_teste}

FORMATO DE SAÍDA:
Responda somente com JSON válido, sem markdown e sem texto adicional.
A resposta deve começar com {{ e terminar com }}.

O JSON deve ter exatamente estes campos:
{{
  "pergunta": "{pergunta_teste}",
  "resposta": "resposta fundamentada no contexto",
  "atores_historicos": ["atores históricos mencionados no contexto"],
  "fontes_recuperadas": ["{arquivo_teste.name}"],
  "nivel_confianca": "alto",
  "limitacoes": "explique brevemente os limites da resposta com base no corpus"
}}

REGRAS:
- Não invente fontes.
- Use apenas o arquivo "{arquivo_teste.name}" em fontes_recuperadas.
- O campo "fontes_recuperadas" deve ser uma lista.
- O campo "nivel_confianca" deve ser apenas "baixo", "médio" ou "alto".
- O campo "limitacoes" é obrigatório.
"""

resposta_estruturada = gerar_resposta(prompt_estruturado, max_tokens=500, temp=0.1)

print(resposta_estruturada)


{
  "pergunta": "Quais processos históricos tornam o período 1836-1936 relevante para a economia moderna?",
  "resposta": "O período de 1836 a 1936 é marcado por um conjunto significativo de transformações que consolidaram e expandiram a industrialização, fundamentando o surgimento da economia moderna. Essas mudanças incluem a mecanização das indústrias têxteis, expansão ferroviária para transporte e comunicação, crescimento do comércio internacional e formação de mercados nacionais integrados.",
  "atores_historicos": ["Reino Unido", "Estados Unidos", "Alemanha", "Japão"],
  "fontes_recuperadas": ["01_contexto_geral_1836_1936.txt"],
  "nivel_confianca": "alto",
  "limitacoes": "As respostas são baseadas no contexto fornecido e não incluem informações adicionais ou expansões que não estejam diretamente relacionadas com os temas de industrialização, tecnologia, capitalismo industrial, urbanização, imperialismo, guerras industriais e crise econômica."
}


In [11]:
avaliacao_estruturada = processar_saida("prompt estruturado", resposta_estruturada)

print("JSON válido:", avaliacao_estruturada["json_valido"])
print("Erros:", avaliacao_estruturada["erros"])

JSON válido: True
Erros: []


## 9. Técnica 3 — Few-shot prompting

Nesta técnica, o modelo recebe exemplos de respostas no formato esperado antes de responder à pergunta final.

In [12]:
prompt_few_shot = f"""
Você deve responder perguntas históricas usando somente JSON válido.

Exemplo 1:
Pergunta: Qual foi o papel das ferrovias na industrialização?
Resposta:
{{
  "pergunta": "Qual foi o papel das ferrovias na industrialização?",
  "resposta": "As ferrovias ampliaram a circulação de matérias-primas, trabalhadores e produtos industriais, integrando mercados e acelerando a expansão econômica.",
  "atores_historicos": ["Reino Unido", "Estados Unidos", "Alemanha"],
  "fontes_recuperadas": ["03_ferrovias_carvao_aco.txt"],
  "nivel_confianca": "alto",
  "limitacoes": "A resposta depende dos documentos disponíveis no corpus."
}}

Exemplo 2:
Pergunta: A União Soviética representou uma alternativa ao capitalismo industrial?
Resposta:
{{
  "pergunta": "A União Soviética representou uma alternativa ao capitalismo industrial?",
  "resposta": "A União Soviética apresentou um modelo alternativo baseado em planejamento estatal, industrialização acelerada e controle público dos meios de produção.",
  "atores_historicos": ["União Soviética"],
  "fontes_recuperadas": ["12_urss_planejamento_industrializacao.txt"],
  "nivel_confianca": "alto",
  "limitacoes": "A resposta resume o tema com base no corpus e não substitui análise historiográfica especializada."
}}

Agora responda à pergunta abaixo seguindo exatamente o mesmo formato dos exemplos.

<contexto>
{contexto}
</contexto>

Pergunta:
{pergunta_teste}

Formato obrigatório da resposta:
{{
  "pergunta": "{pergunta_teste}",
  "resposta": "resposta fundamentada no contexto",
  "atores_historicos": ["lista de atores históricos"],
  "fontes_recuperadas": ["{arquivo_teste.name}"],
  "nivel_confianca": "baixo, médio ou alto",
  "limitacoes": "limitações da resposta com base no corpus"
}}

Regras:
- Responda somente com JSON válido.
- Não use markdown.
- Não use bloco ```json.
- Não escreva nada fora do JSON.
- A resposta deve começar com {{ e terminar com }}.
- Use exatamente "{arquivo_teste.name}" no campo fontes_recuperadas.
- O campo fontes_recuperadas deve ser uma lista.
- O campo atores_historicos deve ser uma lista.
- O campo nivel_confianca deve ser apenas "baixo", "médio" ou "alto".
- O campo limitacoes é obrigatório e não pode ficar vazio.
- Se não houver limitação específica, escreva: "A resposta é limitada ao contexto fornecido pelo corpus."
"""

resposta_few_shot = gerar_resposta(prompt_few_shot, max_tokens=700, temp=0.1)

print(resposta_few_shot)


{
  "pergunta": "Quais processos históricos tornam o período 1836-1936 relevante para a economia moderna?",
  "resposta": "O período de 1836-1936 é marcado pela industrialização, urbanização e expansão do capitalismo que reconfiguraram as bases da organização econômica mundial. Processos como o desenvolvimento das ferrovias, a mecanização nas indústrias têxtil e metalúrgica, a integração de mercados nacionais e internacionais, além do surgimento de novos setores industriais, são fundamentais para entender as transformações da economia moderna.",
  "atores_historicos": ["Reino Unido", "Estados Unidos", "Alemanha", "Japão"],
  "fontes_recuperadas": ["01_contexto_geral_1836_1936.txt"],
  "nivel_confianca": "alto",
  "limitacoes": "A resposta é limitada ao contexto fornecido pelo corpus."
}


In [13]:
avaliacao_few_shot = processar_saida("few-shot", resposta_few_shot)

print("JSON válido:", avaliacao_few_shot["json_valido"])
print("Erros:", avaliacao_few_shot["erros"])

JSON válido: True
Erros: []


## 10. Comparação das técnicas

A comparação considera apenas critérios simples e objetivos:

- se a saída foi JSON válido;
- se todos os campos obrigatórios foram preenchidos;
- se houve erro de parsing ou validação;
- observação qualitativa manual sobre aderência ao formato.

In [14]:
avaliacoes = [
    avaliacao_zero_shot,
    avaliacao_estruturada,
    avaliacao_few_shot
]

linhas = []

for item in avaliacoes:
    dados = item["dados"] or {}
    linhas.append({
        "tecnica": item["tecnica"],
        "json_valido": item["json_valido"],
        "quantidade_erros": len(item["erros"]),
        "erros": "; ".join(item["erros"]),
        "fonte_correta": arquivo_teste.name in dados.get("fontes_recuperadas", []) if isinstance(dados.get("fontes_recuperadas", None), list) else False,
        "pergunta_correta": dados.get("pergunta") == pergunta_teste
    })

df_avaliacao = pd.DataFrame(linhas)
display(df_avaliacao)


resumo_avaliacao = df_avaliacao.to_dict(orient="records")


,tecnica,json_valido,quantidade_erros,erros,fonte_correta,pergunta_correta
0,zero-shot,True,0,,True,True
1,prompt estruturado,True,0,,True,True
2,few-shot,True,0,,True,True


## 11. Salvamento dos resultados

In [15]:
resultados = {
    "data_execucao": datetime.now().isoformat(),
    "modelo": MODEL_NAME,
    "arquivo_contexto": arquivo_teste.name,
    "pergunta_teste": pergunta_teste,
    "device": device_escolhido,
    "tecnicas_prompting": [
        "zero-shot",
        "prompt estruturado com papel, contexto, tarefa e formato",
        "few-shot"
    ],
    "avaliacoes": avaliacoes,
    "resumo_avaliacao": resumo_avaliacao
}

saida_json = OUTPUT_DIR / "c02_prompting_resultados.json"
saida_csv = OUTPUT_DIR / "c02_prompting_avaliacao.csv"

with saida_json.open("w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

df_avaliacao.to_csv(saida_csv, index=False, encoding="utf-8-sig")

print("Resultados salvos em:", saida_json)
print("Resumo salvo em:", saida_csv)


Resultados salvos em: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes\c02_prompting_resultados.json
Resumo salvo em: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes\c02_prompting_avaliacao.csv


## 12. Encerramento do modelo

In [16]:
model.close()
print("Modelo fechado e recursos liberados.")

Modelo fechado e recursos liberados.
